# Semantic Information Retrieval Using WordNet + TF-IDF

## Objective
To develop a semantic information retrieval system that retrieves relevant documents using WordNet-based query expansion and TF-IDF with cosine similarity.

## Technologies Used
- Python
- NLTK WordNet
- Scikit-learn
- Pandas
- Google Colab

## Evaluation Metrics
- Precision
- Recall
- F1 Score
- Average Precision (AP)
- Mean Average Precision (MAP)

Cell 2 — Import libraries and download WordNet

In [1]:
!pip -q install nltk scikit-learn pandas

import nltk
import re
import numpy as np
import pandas as pd

from nltk.corpus import wordnet as wn
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

print("Libraries imported successfully!")
print("WordNet is ready!")

Libraries imported successfully!
WordNet is ready!


Cell 3 — Create the 25-document dataset

In [2]:
documents = [
    "The automobile industry is growing rapidly.",
    "I bought a new car yesterday.",
    "The vehicle requires regular maintenance.",
    "Python is a popular programming language.",
    "The motorcar was invented many years ago.",
    "The computer processes data efficiently.",
    "Automobiles are becoming more environmentally friendly.",
    "The software developer created a new application.",
    "An auto is useful for daily commuting.",
    "A motor vehicle needs fuel and regular servicing.",
    "Public transportation includes buses and trains.",
    "The electronic computer stores and processes information.",
    "A computing device performs calculations quickly.",
    "The data processor handles large amounts of information.",
    "Computer hardware includes memory and processors.",
    "A programmer writes instructions for a computer.",
    "Coding requires logic and problem-solving skills.",
    "The application was developed using software engineering.",
    "The weather was sunny during the picnic.",
    "Fresh vegetables are important for a balanced diet.",
    "The cat slept beside the window.",
    "Football players trained before the match.",
    "The electric vehicle uses a rechargeable battery.",
    "A machine can automate repetitive industrial tasks.",
    "The motorcar engine converts fuel into motion."
]

doc_ids = [f"D{i+1}" for i in range(len(documents))]

print("Total number of documents:", len(documents))
print()

for doc_id, document in zip(doc_ids, documents):
    print(f"{doc_id}: {document}")

Total number of documents: 25

D1: The automobile industry is growing rapidly.
D2: I bought a new car yesterday.
D3: The vehicle requires regular maintenance.
D4: Python is a popular programming language.
D5: The motorcar was invented many years ago.
D6: The computer processes data efficiently.
D7: Automobiles are becoming more environmentally friendly.
D8: The software developer created a new application.
D9: An auto is useful for daily commuting.
D10: A motor vehicle needs fuel and regular servicing.
D11: Public transportation includes buses and trains.
D12: The electronic computer stores and processes information.
D13: A computing device performs calculations quickly.
D14: The data processor handles large amounts of information.
D15: Computer hardware includes memory and processors.
D16: A programmer writes instructions for a computer.
D17: Coding requires logic and problem-solving skills.
D18: The application was developed using software engineering.
D19: The weather was sunny duri

Cell 4 — Text preprocessing and WordNet query expansion

In [3]:
lemmatizer = WordNetLemmatizer()


def preprocess_text(text):
    tokens = re.findall(r"[a-zA-Z]+", text.lower())

    processed_tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
    ]

    return " ".join(processed_tokens)


def expand_query_semantically(query):
    expanded_terms = set(preprocess_text(query).split())

    words = preprocess_text(query).split()

    for word in words:
        synsets = wn.synsets(word, pos=wn.NOUN)

        if synsets:
            # Use the first noun sense
            main_synset = synsets[0]

            # Add synonyms
            for lemma in main_synset.lemmas():
                term = lemma.name().replace("_", " ")
                expanded_terms.update(preprocess_text(term).split())

            # Add hypernym terms
            for hypernym in main_synset.hypernyms():
                for lemma in hypernym.lemmas():
                    term = lemma.name().replace("_", " ")
                    expanded_terms.update(preprocess_text(term).split())

    return sorted(expanded_terms)


query = "car"

print("Original Query:", query)
print("Expanded Query:", expand_query_semantically(query))

print("\nPreprocessing Example:")
print("Before:", "Automobiles are becoming more environmentally friendly.")
print("After: ", preprocess_text(
    "Automobiles are becoming more environmentally friendly."
))

Original Query: car
Expanded Query: ['auto', 'automobile', 'automotive', 'car', 'machine', 'motor', 'motorcar', 'vehicle']

Preprocessing Example:
Before: Automobiles are becoming more environmentally friendly.
After:  automobile are becoming more environmentally friendly


Cell 5 — TF-IDF and semantic retrieval

In [4]:
# Preprocess the document collection
processed_documents = [
    preprocess_text(document)
    for document in documents
]

# Create TF-IDF vectors
vectorizer = TfidfVectorizer()
document_vectors = vectorizer.fit_transform(processed_documents)

print("Number of documents:", document_vectors.shape[0])
print("Number of TF-IDF features:", document_vectors.shape[1])


def retrieve_documents(query):
    # Expand the query using WordNet
    expanded_terms = expand_query_semantically(query)
    expanded_query = " ".join(expanded_terms)

    # Preprocess and vectorize the expanded query
    processed_query = preprocess_text(expanded_query)
    query_vector = vectorizer.transform([processed_query])

    # Calculate cosine similarity
    similarities = cosine_similarity(
        query_vector, document_vectors
    ).flatten()

    # Rank documents by similarity
    ranked_indices = np.argsort(-similarities, kind="stable")

    ranked_results = [
        {
            "Document ID": doc_ids[i],
            "Similarity": similarities[i],
            "Document": documents[i]
        }
        for i in ranked_indices
        if similarities[i] > 0
    ]

    return expanded_query, ranked_results, similarities

Number of documents: 25
Number of TF-IDF features: 111


Cell 6 — Demonstrate semantic search

In [5]:
query = "car"

expanded_query, ranked_results, car_similarities = retrieve_documents(query)

print("Original Query:", query)
print("Expanded Query:", expanded_query)

print("\nRanked Semantic Search Results:\n")

for rank, result in enumerate(ranked_results, start=1):
    print(
        f"{rank}. {result['Document ID']} "
        f"(Score: {result['Similarity']:.4f})"
    )
    print("   ", result["Document"])

Original Query: car
Expanded Query: auto automobile automotive car machine motor motorcar vehicle

Ranked Semantic Search Results:

1. D10 (Score: 0.2770)
    A motor vehicle needs fuel and regular servicing.
2. D2 (Score: 0.2061)
    I bought a new car yesterday.
3. D24 (Score: 0.1637)
    A machine can automate repetitive industrial tasks.
4. D9 (Score: 0.1598)
    An auto is useful for daily commuting.
5. D1 (Score: 0.1462)
    The automobile industry is growing rapidly.
6. D3 (Score: 0.1406)
    The vehicle requires regular maintenance.
7. D7 (Score: 0.1334)
    Automobiles are becoming more environmentally friendly.
8. D5 (Score: 0.1326)
    The motorcar was invented many years ago.
9. D25 (Score: 0.1310)
    The motorcar engine converts fuel into motion.
10. D23 (Score: 0.1181)
    The electric vehicle uses a rechargeable battery.


Cell 7 — Evaluation metrics

In [6]:
def calculate_metrics(ranked_results, similarities, relevant_documents):
    # Only documents with positive similarity are retrieved
    retrieved_documents = {
        result["Document ID"]
        for result in ranked_results
    }

    relevant_documents = set(relevant_documents)

    true_positives = len(
        retrieved_documents & relevant_documents
    )

    precision = (
        true_positives / len(retrieved_documents)
        if retrieved_documents else 0.0
    )

    recall = (
        true_positives / len(relevant_documents)
        if relevant_documents else 0.0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    return precision, recall, f1


def calculate_ap(ranked_results, relevant_documents):
    relevant_documents = set(relevant_documents)

    if not relevant_documents:
        return 0.0

    relevant_count = 0
    precision_sum = 0.0

    for rank, result in enumerate(ranked_results, start=1):
        if result["Document ID"] in relevant_documents:
            relevant_count += 1
            precision_sum += relevant_count / rank

    # Divide by the total number of relevant documents,
    # including relevant documents that were not retrieved.
    return precision_sum / len(relevant_documents)

In [18]:
# Define relevant documents for the query "car"
relevant_car_docs = [
    "D1", "D2", "D3", "D5", "D7",
    "D9", "D10", "D23", "D25"
]

# Calculate evaluation metrics
precision, recall, f1 = calculate_metrics(
    ranked_results,
    car_similarities,
    relevant_car_docs
)

average_precision = calculate_ap(
    ranked_results,
    relevant_car_docs
)

print("Evaluation Results for Query:", query)
print("-----------------------------------")
print(f"Precision:         {precision:.4f}")
print(f"Recall:            {recall:.4f}")
print(f"F1 Score:          {f1:.4f}")
print(f"Average Precision: {average_precision:.4f}")

Evaluation Results for Query: programming
-----------------------------------
Precision:         0.9000
Recall:            1.0000
F1 Score:          0.9474
Average Precision: 0.8783


In [19]:
# Test the system with multiple queries

test_queries = {
    "car": [
        "D1", "D2", "D3", "D5", "D7",
        "D9", "D10", "D23", "D25"
    ],
    "computer": [
        "D6", "D12", "D13", "D14", "D15", "D16"
    ],
    "programming": [
        "D4", "D16", "D17", "D18"
    ]
}

print("Updated relevance labels successfully.")


evaluation_results = []

for test_query, relevant_docs in test_queries.items():
    expanded_query, results, similarities = retrieve_documents(test_query)

    precision, recall, f1 = calculate_metrics(
        results, similarities, relevant_docs
    )

    ap = calculate_ap(results, relevant_docs)

    evaluation_results.append({
        "Query": test_query,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Average Precision": ap
    })

evaluation_df = pd.DataFrame(evaluation_results)

print("Evaluation Results for Multiple Queries")
display(evaluation_df.round(4))

Updated relevance labels successfully.
Evaluation Results for Multiple Queries


,Query,Precision,Recall,F1 Score,Average Precision
0,car,0.9000,1.00,0.9474,0.8783
1,computer,0.8571,1.00,0.9231,0.9762
2,programming,1.0000,0.25,0.4000,0.2500


Cell 10 — Calculate MAP

In [20]:
# Calculate Mean Average Precision (MAP)
map_score = evaluation_df["Average Precision"].mean()

print("Mean Average Precision (MAP):", round(map_score, 4))

print("\nFinal Evaluation Summary")
print("------------------------")
print("Number of test queries:", len(test_queries))
print(f"Mean Precision: {evaluation_df['Precision'].mean():.4f}")
print(f"Mean Recall:    {evaluation_df['Recall'].mean():.4f}")
print(f"Mean F1 Score:  {evaluation_df['F1 Score'].mean():.4f}")
print(f"MAP Score:      {map_score:.4f}")

Mean Average Precision (MAP): 0.7015

Final Evaluation Summary
------------------------
Number of test queries: 3
Mean Precision: 0.9190
Mean Recall:    0.7500
Mean F1 Score:  0.7568
MAP Score:      0.7015


Cell 11: Interactive search demo

In [21]:

# Interactive Semantic Search + Evaluation

user_query = input("Enter your search query: ").strip()

expanded_query, results, similarities = retrieve_documents(user_query)

print("\n" + "=" * 60)
print("SEMANTIC INFORMATION RETRIEVAL RESULTS")
print("=" * 60)

print("Original Query:", user_query)
print("Expanded Query:", expanded_query)

# Display ranked documents
print("\nRANKED SEARCH RESULTS")
print("-" * 60)

if results:
    for rank, result in enumerate(results, start=1):
        print(
            f"{rank}. {result['Document ID']} "
            f"(Similarity: {result['Similarity']:.4f})"
        )
        print("   " + result["Document"])
else:
    print("No matching documents found.")

# Evaluate only queries for which we have ground-truth labels
query_key = user_query.lower()

if query_key in test_queries:
    relevant_docs = test_queries[query_key]

    precision, recall, f1 = calculate_metrics(
        results, similarities, relevant_docs
    )

    ap = calculate_ap(results, relevant_docs)

    print("\nEVALUATION METRICS")
    print("-" * 60)
    print(f"Precision:         {precision:.4f}")
    print(f"Recall:            {recall:.4f}")
    print(f"F1 Score:          {f1:.4f}")
    print(f"Average Precision: {ap:.4f}")
    print(f"MAP (all 3 queries): {map_score:.4f}")

    print("\nRelevant documents:", ", ".join(relevant_docs))
    print("Relevant documents retrieved:",
          len(set(r["Document ID"] for r in results)
              & set(relevant_docs)))
else:
    print("\nEvaluation metrics are unavailable for this query.")
    print("Add relevance labels for this query to test its metrics.")
    print("MAP remains based on the three predefined test queries.")


Enter your search query: car

SEMANTIC INFORMATION RETRIEVAL RESULTS
Original Query: car
Expanded Query: auto automobile automotive car machine motor motorcar vehicle

RANKED SEARCH RESULTS
------------------------------------------------------------
1. D10 (Similarity: 0.2770)
   A motor vehicle needs fuel and regular servicing.
2. D2 (Similarity: 0.2061)
   I bought a new car yesterday.
3. D24 (Similarity: 0.1637)
   A machine can automate repetitive industrial tasks.
4. D9 (Similarity: 0.1598)
   An auto is useful for daily commuting.
5. D1 (Similarity: 0.1462)
   The automobile industry is growing rapidly.
6. D3 (Similarity: 0.1406)
   The vehicle requires regular maintenance.
7. D7 (Similarity: 0.1334)
   Automobiles are becoming more environmentally friendly.
8. D5 (Similarity: 0.1326)
   The motorcar was invented many years ago.
9. D25 (Similarity: 0.1310)
   The motorcar engine converts fuel into motion.
10. D23 (Similarity: 0.1181)
   The electric vehicle uses a rechargeable b

## Conclusion

This project successfully implements a semantic information retrieval system using WordNet-based query expansion, TF-IDF vectorization, and cosine similarity.

The system was tested on a collection of 25 documents using three queries: **car, computer, and programming**.

### Evaluation Results

- Mean Precision: **0.9190**
- Mean Recall: **0.7500**
- Mean F1 Score: **0.7568**
- Mean Average Precision (MAP): **0.7015**

The results indicate that the system achieves high precision but has difficulty retrieving all relevant documents for certain queries, particularly programming.

### Limitations

- The dataset contains only 25 manually created documents.
- WordNet expansion may introduce unrelated terms or miss useful synonyms.
- TF-IDF relies on term overlap and cannot fully understand semantic meaning.
- Evaluation is based on manually assigned relevance labels.
